In [28]:
import pyquil
import numpy as np
from pyquil import Program, get_qc
from pyquil.gates import H, RX, RZ, CNOT, MEASURE, X
import collections


In [29]:
qc = get_qc("Ankaa-3")

In [32]:
# ---- PARAMETERS ----
gamma = 3.20712571
beta  = 5.37330497

# ---- SUBGRAPH EDGES ----
sub_edges = [
    (0, 95, 27.793273849853),
    (7, 149, 43.877887232445),
    (7, 173, 18.474661518371),
    (14, 79, 45.881110087758),
    (14, 90, 68.956466480011),
    (18, 95, 18.994044127633),
    (42, 90, 40.658020198262),
    (42, 95, 14.614253060312),
    (47, 95, 14.319923416947),
    (79, 168, 45.711427645228),
    (82, 95, 14.688691199195),
    (82, 107, 34.211122042632),
    (87, 90, 119.895996514071),
    (87, 110, 41.050179744646),
    (87, 164, 57.351330892876),
    (87, 173, 29.912609867151),
    (90, 107, 32.266915721732),
    (90, 167, 41.218956122872),
    (95, 138, 25.312244674362),
    (95, 164, 14.56316198281),
    (110, 173, 43.351650424353),
    (149, 167, 50.518391305245),
    (164, 168, 41.412353277444),
    (167, 178, 28.71807250327)
]

# ---- NODE TO QUBIT MAPPING ----
node_to_qubit = {
    0:0, 7:1, 14:2, 18:3, 42:4, 47:5,
    79:6, 82:7, 87:8, 90:9, 95:10,
    107:11, 110:12, 138:13, 149:14,
    164:15, 167:16, 168:17, 173:18, 178:19
}

num_qubits = 20

# ---- PROGRAM ----
p = Program()

# Initial |+> state
for q in range(num_qubits):
    p += X(q)
    p += H(q)

# Cost layer
for (u, v, w) in sub_edges:
    i = node_to_qubit[u]
    j = node_to_qubit[v]
    
    theta = 2 * gamma * w
    
    p += CNOT(i, j)
    p += RZ(theta, j)
    p += CNOT(i, j)

# Mixer layer
for q in range(num_qubits):
    p += RX(2 * beta, q)

# Add measurements
ro = p.declare("ro", "BIT", num_qubits)
for q in range(num_qubits):
    p += MEASURE(q, ro[q])


print(p)

DECLARE ro BIT[20]
X 0
H 0
X 1
H 1
X 2
H 2
X 3
H 3
X 4
H 4
X 5
H 5
X 6
H 6
X 7
H 7
X 8
H 8
X 9
H 9
X 10
H 10
X 11
H 11
X 12
H 12
X 13
H 13
X 14
H 14
X 15
H 15
X 16
H 16
X 17
H 17
X 18
H 18
X 19
H 19
CNOT 0 10
RZ(178.2730462578685) 10
CNOT 0 10
CNOT 1 14
RZ(281.4438004873102) 14
CNOT 1 14
CNOT 1 18
RZ(118.50112387823056) 18
CNOT 1 18
CNOT 2 6
RZ(294.2929755315781) 6
CNOT 2 6
CNOT 2 9
RZ(442.30411303759297) 9
CNOT 2 9
CNOT 3 10
RZ(121.83257451721263) 10
CNOT 3 10
CNOT 4 9
RZ(260.79076379109074) 9
CNOT 4 9
CNOT 4 10
RZ(93.7394934443456) 10
CNOT 4 10
CNOT 5 10
RZ(91.85158911144356) 10
CNOT 5 10
CNOT 6 17
RZ(293.20458968363096) 17
CNOT 6 17
CNOT 7 10
RZ(94.21695838237804) 10
CNOT 7 10
CNOT 7 11
RZ(219.4387381417456) 11
CNOT 7 11
CNOT 8 9
RZ(769.043065892695) 9
CNOT 8 9
CNOT 8 12
RZ(263.30617371835086) 12
CNOT 8 12
CNOT 8 15
RZ(367.8658556185198) 15
CNOT 8 15
CNOT 8 18
RZ(191.86700031627933) 18
CNOT 8 18
CNOT 9 11
RZ(206.9681099871398) 11
CNOT 9 11
CNOT 9 16
RZ(264.38874784204944) 16
CNOT 9 

In [40]:
# Expectation Function 
def compute_expectation(results, sub_edges, node_to_qubit):
    exp_val = 0
    
    for (u, v, w) in sub_edges:
        i = node_to_qubit[u]
        j = node_to_qubit[v]
        
        zij = 0
        for shot in results:
            zi = (1 - 2*shot[i] - (p0[i] - (1-p1[i])))
            zj = 1 if shot[j] == 0 else -1
            zij += zi * zj
        
        zij /= len(results)
        exp_val += -0.5 * w * zij
    
    return exp_val


In [41]:
def run_on_qpu(shots):
    prog = p.copy()
    prog.wrap_in_numshots_loop(shots)
    
    compiled = qc.compile(prog)
    result = qc.run(compiled)
    
    results_array = np.array(result.readout_data["ro"])
    
    energy = compute_expectation(results_array, sub_edges, node_to_qubit)
    
    return energy


In [42]:
for shots in [1000, 5000, 10000]:
    energy = run_on_qpu(shots)
    print(f"Shots: {shots} -> Energy: {energy}")

/tmp/ipykernel_197/305184613.py:8: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  results_array = np.array(result.readout_data["ro"])


Shots: 1000 -> Energy: -5.508401635923835
Shots: 5000 -> Energy: -4.720138337908059
Shots: 10000 -> Energy: -13.899816044223709


In [43]:
#  Readout Calibration |0> 
calib_prog_0 = Program()
ro = calib_prog_0.declare("ro", "BIT", num_qubits)

for q in range(num_qubits):
    calib_prog_0 += MEASURE(q, ro[q])

calib_prog_0.wrap_in_numshots_loop(2000)

compiled_0 = qc.compile(calib_prog_0)
result_0 = qc.run(compiled_0)

calib_0 = np.array(result_0.readout_data["ro"])

/tmp/ipykernel_197/4250427146.py:13: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  calib_0 = np.array(result_0.readout_data["ro"])


In [44]:
calib_prog_1 = Program()
ro = calib_prog_1.declare("ro", "BIT", num_qubits)

# prepare |1> on all qubits first
for q in range(num_qubits):
    calib_prog_1 += RX(np.pi, q)

# then measure all
for q in range(num_qubits):
    calib_prog_1 += MEASURE(q, ro[q])

calib_prog_1.wrap_in_numshots_loop(2000)

compiled_1 = qc.compile(calib_prog_1)
result_1 = qc.run(compiled_1)

calib_1 = np.array(result_1.readout_data["ro"])

/tmp/ipykernel_197/1957702249.py:17: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  calib_1 = np.array(result_1.readout_data["ro"])


In [45]:
p0 = np.mean(calib_0, axis=0)      # measured probability of 1 when prepared 0
p1 = np.mean(calib_1, axis=0)      # measured probability of 1 when prepared 1

print("Readout error per qubit:")
print("P(1|0):", p0)
print("P(1|1):", p1)

Readout error per qubit:
P(1|0): [0.032  0.033  0.0125 0.0085 0.0105 0.0345 0.0075 0.0095 0.006  0.023
 0.0315 0.016  0.014  0.0115 0.0185 0.047  0.0155 0.023  0.009  0.015 ]
P(1|1): [0.9515 0.943  0.96   0.964  0.9725 0.9315 0.953  0.9695 0.9685 0.968
 0.88   0.9675 0.959  0.9035 0.924  0.967  0.98   0.937  0.9715 0.9095]
